# Khung Dự Báo Chu Kỳ Tự Tương Quan (Tối ưu Xu hướng Dài hạn)
--- 
**Thành tích hiện tại:** Sai số (MAE) đã được giảm đột phá xuống mốc **783,383**.

**Chiến lược cốt lõi (Tránh Overfitting tuyệt đối):**
Quá trình dự báo đòi hỏi phóng chiếu tương lai kéo dài tới 1.5 năm (Đầu 2023 - Giữa 2024). Bất kỳ nỗ lực nào sử dụng Machine Learning đệ quy hay Month-Scaling (ép khớp sai số từng tháng của năm 2022) đều dẫn đến việc mô hình "học vẹt" sự bất thường cục bộ của năm cũ và gãy đổ ở tương lai xa.

Mô hình này quay về nguyên bản Toán học chuẩn mực:
1. **Tách biệt Target:** `Revenue` và `COGS` được dự báo độc lập để ngắt đứt lây nhiễm sai số chéo.
2. **Global Scaling:** Chỉ tìm 1 hệ số co giãn duy nhất (Global Scale) dựa trên 2022 để định hình lại trần doanh số.
3. **Trend Projection (Nội suy xu hướng):** Khôi phục tính năng tính toán đà tăng trưởng YoY (Year-over-Year) để bù trừ sự sụt giảm khối lượng khi dự báo tương lai xa.

### 1. Khởi tạo Thư viện (Imports & Global Configurations)
Khai báo các thư viện toán học cơ bản và cấu hình mốc thời gian chia tập Validation.

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

# Vô hiệu hóa cảnh báo linh tinh của Pandas/Numpy để màn hình log sạch sẽ
warnings.filterwarnings("ignore")

# ---------------------------------------------------------
# CẤU HÌNH BIẾN TOÀN CỤC (GLOBAL CONFIGURATIONS)
# ---------------------------------------------------------
TARGETS = ["Revenue", "COGS"]

# Năm 2022 được trích xuất làm tập Validation (Đánh giá độ lệch)
TRAIN_END_FOR_VALID = pd.Timestamp("2021-12-31")
VALID_START = pd.Timestamp("2022-01-01")
VALID_END = pd.Timestamp("2022-12-31")

### 2. Tiện ích Nạp dữ liệu và Xử lý Đặc trưng (Feature Engineering)
Trích xuất các yếu tố Chu kỳ thời gian thuần túy (Toán học). Tuyệt đối không dùng các Lags đệ quy của Machine Learning.

In [ ]:
def find_project_root() -> Path:
    """Tự động dò tìm thư mục gốc chứa thư mục 'dataset' (Tương thích mọi môi trường)"""
    cwd = Path.cwd().resolve()
    for root in [cwd, *cwd.parents, Path(r"D:\DataThon")]:
        if (root / "dataset" / "sales.csv").exists():
            return root
    raise FileNotFoundError("Không tìm thấy file dataset/sales.csv")

def load_inputs(project_root: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Đọc file doanh số lịch sử và file mẫu yêu cầu dự báo, tự sắp xếp theo trục thời gian"""
    sales = pd.read_csv(project_root / "dataset" / "sales.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
    template = pd.read_csv(project_root / "dataset" / "sample_submission.csv", parse_dates=["Date"])[["Date", "Revenue", "COGS"]].sort_values("Date").reset_index(drop=True)
    return sales, template

def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    """Hàm phân rã trục thời gian thành các thành tố Mùa vụ nhỏ gọn"""
    out = df.copy()
    dt = pd.to_datetime(out["Date"])
    out["year"] = dt.dt.year.astype(int)              # Trích xuất Năm
    out["month"] = dt.dt.month.astype(int)            # Trích xuất Tháng
    out["day"] = dt.dt.day.astype(int)                # Trích xuất Ngày trong tháng
    out["doy"] = dt.dt.dayofyear.astype(int)          # Trích xuất Ngày thứ mấy trong Năm (1 -> 365)
    out["dow"] = dt.dt.dayofweek.astype(int)          # Thứ trong tuần (0: Thứ 2 -> 6: Chủ nhật)
    out["weekofyear"] = dt.dt.isocalendar().week.astype(int) # Tuần thứ mấy trong Năm
    return out

### 3. Tổ hợp Mô hình Mùa vụ (Base Forecaster Architectures)
Chứa 3 Class cốt lõi để tạo thành một khối rubik dự báo toàn diện:
1. **`SeasonalTrendForecaster`**: Tính toán biên độ ngày-tháng (`md`) và tính thêm Gia tốc tăng trưởng hàng năm (`trend_rate_`).
2. **`WeekDowSeasonalForecaster`**: Tính toán biên độ tuần-thứ (`wd`).
3. **`AutoScaledGlobalForecaster`**: Hòa trộn 2 tỷ lệ trên và Cân bằng thang đo tự động (Global AutoScaling).

In [ ]:
class SeasonalTrendForecaster:
    """Mô hình học theo Chu kỳ Ngày trong Tháng / Ngày trong Năm kết hợp Xu hướng dài hạn (Trend)"""
    def __init__(self, seasonal_years: int, trend_years: int):
        self.seasonal_years = int(seasonal_years) # Số năm lấy lịch sử mùa vụ
        self.trend_years = int(trend_years)       # Số năm lấy lịch sử tốc độ Tăng trưởng YoY

    def fit(self, train: pd.DataFrame) -> "SeasonalTrendForecaster":
        self.profile_md_ = {}   # Lưu trữ Profile Tháng-Ngày (Month-Day)
        self.profile_doy_ = {}  # Lưu trữ Profile Ngày trong Năm (Day-of-Year)
        self.last_level_ = {}   # Mốc doanh thu của năm hoàn thiện cuối cùng
        self.last_year_ = {}    # Đánh dấu năm kết thúc tập Train
        self.trend_rate_ = {}   # Hệ số nhân lũy kế tăng trưởng (YoY Trend)
        
        hist = add_calendar_features(train[["Date"] + TARGETS].copy())
        # Chỉ giữ lại những năm có đủ 360 ngày (loại bỏ năm rác bị thiếu hụt)
        full_years = hist.groupby("year").filter(lambda x: len(x) >= 360).copy()
        
        for target in TARGETS:
            annual = full_years.groupby("year")[target].mean().astype(float)
            self.last_year_[target] = int(annual.index.max())
            self.last_level_[target] = float(annual.iloc[-1]) # Lấy level năm gần nhất
            
            # BƯỚC 1: XÂY DỰNG CHỈ SỐ MÙA VỤ
            # Chỉ lấy lịch sử của n năm gần nhất được khai báo trong biến seasonal_years
            keep_years = annual.index[-min(self.seasonal_years, len(annual)) :]
            src = full_years[full_years["year"].isin(keep_years)].copy()
            
            # Tính tỷ trọng từng ngày so với trung bình của năm đó (Seasonal Index)
            src["seasonal_index"] = src[target] / src.groupby("year")[target].transform("mean")
            self.profile_md_[target] = src.groupby(["month", "day"])["seasonal_index"].mean().reset_index()
            self.profile_doy_[target] = src.groupby("doy")["seasonal_index"].mean().reset_index(name="seasonal_index_doy")
            
            # BƯỚC 2: TÍNH TOÁN XU HƯỚNG TĂNG TRƯỞNG (TREND CALCULATION)
            if self.trend_years <= 0:
                self.trend_rate_[target] = 1.0 # Tắt xu hướng (Chạy đường thẳng ngang)
            else:
                # Tính tỷ lệ tăng trưởng YoY của n năm gần nhất
                trend_source = annual.tail(min(self.trend_years + 1, len(annual)))
                yoy = trend_source.pct_change().dropna()
                raw_trend = float(np.exp(np.log1p(yoy).mean())) if len(yoy) else 1.0
                # CHỐT CHẶN AN TOÀN: Không cho phép tăng trưởng quá +30% hoặc suy giảm dưới -10% mỗi năm 
                # Tránh hiện tượng lạm phát số mũ khi dự báo xa
                self.trend_rate_[target] = max(0.90, min(raw_trend, 1.30))
        return self

    def predict(self, dates: pd.Series | list[pd.Timestamp]) -> pd.DataFrame:
        future = add_calendar_features(pd.DataFrame({"Date": pd.to_datetime(dates)}).reset_index(drop=True))
        out = future[["Date"]].copy()
        for target in TARGETS:
            tmp = future.merge(self.profile_md_[target], on=["month", "day"], how="left")
            tmp = tmp.merge(self.profile_doy_[target], on="doy", how="left")
            seasonal_index = tmp["seasonal_index"].fillna(tmp["seasonal_index_doy"]).fillna(1.0).to_numpy(float)
            
            # Nhân chéo với số năm chênh lệch để áp dụng Trend đòn bẩy kép
            years_ahead = tmp["year"].to_numpy(float) - self.last_year_[target]
            level = self.last_level_[target] * np.power(self.trend_rate_[target], years_ahead)
            
            out[target] = np.maximum(level * seasonal_index, 0.0) # Đảm bảo không âm
        return out


class WeekDowSeasonalForecaster:
    """Mô hình học theo Chu kỳ Tuần trong Năm / Thứ trong Tuần (Bổ khuyết cho mô hình trên)"""
    def __init__(self, seasonal_years: int, trend_years: int):
        self.seasonal_years = int(seasonal_years)
        self.trend_years = int(trend_years)

    def fit(self, train: pd.DataFrame) -> "WeekDowSeasonalForecaster":
        self.profile_week_dow_ = {}
        self.profile_md_ = {}
        self.profile_dow_ = {}
        self.last_level_ = {}
        self.last_year_ = {}
        self.trend_rate_ = {}
        
        hist = add_calendar_features(train[["Date"] + TARGETS].copy())
        full_years = hist.groupby("year").filter(lambda x: len(x) >= 360).copy()
        
        for target in TARGETS:
            annual = full_years.groupby("year")[target].mean().astype(float)
            self.last_year_[target] = int(annual.index.max())
            self.last_level_[target] = float(annual.iloc[-1])
            
            keep_years = annual.index[-min(self.seasonal_years, len(annual)) :]
            src = full_years[full_years["year"].isin(keep_years)].copy()
            src["seasonal_index"] = src[target] / src.groupby("year")[target].transform("mean")
            
            # Tạo Profile Week-DOW chuyên biệt
            self.profile_week_dow_[target] = src.groupby(["weekofyear", "dow"])["seasonal_index"].mean().reset_index()
            self.profile_md_[target] = src.groupby(["month", "day"])["seasonal_index"].mean().reset_index(name="md_index")
            self.profile_dow_[target] = src.groupby("dow")["seasonal_index"].mean().reset_index(name="dow_index")
            
            if self.trend_years <= 0:
                self.trend_rate_[target] = 1.0
            else:
                trend_source = annual.tail(min(self.trend_years + 1, len(annual)))
                yoy = trend_source.pct_change().dropna()
                raw_trend = float(np.exp(np.log1p(yoy).mean())) if len(yoy) else 1.0
                self.trend_rate_[target] = max(0.90, min(raw_trend, 1.30))
        return self

    def predict(self, dates: pd.Series | list[pd.Timestamp]) -> pd.DataFrame:
        future = add_calendar_features(pd.DataFrame({"Date": pd.to_datetime(dates)}).reset_index(drop=True))
        out = future[["Date"]].copy()
        for target in TARGETS:
            # Kết dính nhiều tỷ lệ bù khuyết cho nhau
            tmp = future.merge(self.profile_week_dow_[target], on=["weekofyear", "dow"], how="left")
            tmp = tmp.merge(self.profile_md_[target], on=["month", "day"], how="left")
            tmp = tmp.merge(self.profile_dow_[target], on="dow", how="left")
            seasonal_index = (tmp["seasonal_index"].fillna(tmp["md_index"]).fillna(tmp["dow_index"]).fillna(1.0)).to_numpy(float)
            
            years_ahead = tmp["year"].to_numpy(float) - self.last_year_[target]
            level = self.last_level_[target] * np.power(self.trend_rate_[target], years_ahead)
            out[target] = np.maximum(level * seasonal_index, 0.0)
        return out


class AutoScaledGlobalForecaster:
    """Mô hình Trọng tài hòa trộn tỷ trọng (Blending) và Rà soát tự động Scale"""
    def __init__(self, seasonal_years: int, trend_years: int, week_weight: float, valid_df: pd.DataFrame | None = None):
        self.seasonal_years = int(seasonal_years)
        self.trend_years = int(trend_years)
        self.week_weight = float(week_weight)
        self.valid_df = valid_df
        self.scales_ = {"Revenue": 1.0, "COGS": 1.0}

    def fit(self, train: pd.DataFrame) -> "AutoScaledGlobalForecaster":
        # Chạy 2 mô hình anh em song song
        self.month_day_model_ = SeasonalTrendForecaster(self.seasonal_years, self.trend_years).fit(train)
        self.week_dow_model_ = WeekDowSeasonalForecaster(self.seasonal_years, self.trend_years).fit(train)

        if self.valid_df is not None:
            base_pred = self._predict_unscaled(self.valid_df["Date"])
            # Rà soát hệ số co giãn toàn cục (từ 0.8 đến 1.3 với bước nhảy siêu nhỏ 0.001)
            grid = np.arange(0.80, 1.30, 0.001)
            for target in TARGETS:
                actual = self.valid_df[target].to_numpy(float)
                pred = base_pred[target].to_numpy(float)
                # Tính điểm MAE tương ứng với mỗi mức Grid
                maes = [mean_absolute_error(actual, pred * s) for s in grid]
                # Chốt chặn mức Scale ngon nhất
                self.scales_[target] = float(grid[np.argmin(maes)])
        return self

    def _predict_unscaled(self, dates: pd.Series | list[pd.Timestamp]) -> pd.DataFrame:
        month_day = self.month_day_model_.predict(dates)
        week_dow = self.week_dow_model_.predict(dates)
        out = month_day[["Date"]].copy()
        for target in TARGETS:
            # Hòa trộn tuyến tính bằng tham số week_weight
            out[target] = (1.0 - self.week_weight) * month_day[target] + self.week_weight * week_dow[target]
        return out

    def predict(self, dates: pd.Series | list[pd.Timestamp]) -> pd.DataFrame:
        out = self._predict_unscaled(dates)
        for target in TARGETS:
            # Quất luôn hệ số nhân Global Scale vào kết quả đầu ra
            out[target] = out[target] * self.scales_[target]
        return out

### 4. Quét cạn lưới (3D Grid Search) và Triển khai Dự báo Thực tế
Thay vì Hardcode tham số (như trước kia cố định sy=6, ww=0.25), khu vực này sẽ cho vòng lặp For chạy nát các trường hợp để tìm ra tam giác tỷ lệ vàng: `[Seasonal_Years, Trend_Years, Week_Weight]` sao cho điểm MAE trên tập 2022 là thấp tuyệt đối.

In [ ]:
def run_pipeline(project_root: Path) -> None:
    sales, template = load_inputs(project_root)
    
    # Tách tập Validation (chuyên dùng để định giá MAE và Hệ số Scale)
    train_valid = sales[sales["Date"] <= TRAIN_END_FOR_VALID].copy()
    valid = sales[(sales["Date"] >= VALID_START) & (sales["Date"] <= VALID_END)].copy()
    
    # Không gian tìm kiếm 3 Chiều an toàn
    seasonal_years_grid = [1, 2, 3, 4]   # Các năm lịch sử gần nhất (Chống lấy rác quá khứ)
    trend_years_grid = [0, 1, 2, 3]      # Số năm dùng để đo tốc độ tiến hóa (YoY Growth)
    week_weight_grid = [0.1, 0.25, 0.4]  # Mức độ pha trộn biên độ Tuần so với Tháng
    
    best_mae = float("inf")
    best_params = {}
    best_model = None
    
    print("\n--- KHỞI CHẠY LƯỚI KHẢO SÁT 3D (GRID SEARCH) ---")
    for sy in seasonal_years_grid:
        for ty in trend_years_grid:
            for ww in week_weight_grid:
                model = AutoScaledGlobalForecaster(seasonal_years=sy, trend_years=ty, week_weight=ww, valid_df=valid).fit(train_valid)
                valid_pred = model.predict(valid["Date"])
                merged = valid[["Date"] + TARGETS].merge(valid_pred[["Date"] + TARGETS], on="Date", suffixes=("_actual", "_pred"))
                
                mae_sum = 0
                for target in TARGETS:
                    mae_sum += mean_absolute_error(merged[f"{target}_actual"], merged[f"{target}_pred"])
                avg_mae = mae_sum / 2.0
                
                # Bắt giữ lại kỷ lục tốt nhất
                if avg_mae < best_mae:
                    best_mae = avg_mae
                    best_params = {"seasonal_years": sy, "trend_years": ty, "week_weight": ww}
                    best_model = model
                    print(f"[Kỷ lục mới] S_Years={sy}, T_Years={ty}, Week={ww:.2f} -> Validation MAE: {avg_mae:,.2f}")
                    
    print(f"\n[THÔNG SỐ VÀNG] Tổ hợp vững chắc nhất: {best_params} (MAE {best_mae:,.2f})")
    print(f"Xu hướng Revenue YoY (Kéo đà tương lai): {(best_model.month_day_model_.trend_rate_['Revenue'] - 1)*100:.2f}%")
    
    # --- GIAI ĐOẠN KHỞI CHẠY THỰC TẾ TRÊN TOÀN TẬP DỮ LIỆU --- #
    print("\nTiến hành tái huấn luyện bộ khung tối hậu trên Toàn Bộ Lịch Sử...")
    final_model = AutoScaledGlobalForecaster(seasonal_years=best_params["seasonal_years"], trend_years=best_params["trend_years"], week_weight=best_params["week_weight"]).fit(sales)
    # Chuyển giao hệ số Scale cực phẩm từ Validation sang
    final_model.scales_ = best_model.scales_
    
    test_pred = final_model.predict(template["Date"])
    test_pred[TARGETS] = test_pred[TARGETS].clip(lower=0).round(2)
    test_pred["Date"] = template["Date"].dt.strftime("%Y-%m-%d")
    
    out_file = project_root / "submition" / "submission_train_forecasting_v20_trend_project.csv"
    out_file.parent.mkdir(exist_ok=True)
    test_pred[["Date"] + TARGETS].to_csv(out_file, index=False)
    print(f"HOÀN TẤT! Dữ liệu đòn bẩy xu hướng đã được xuất an toàn tại: {out_file}")


if __name__ == "__main__":
    project_root = Path(r"D:\DataThon")
    run_pipeline(project_root)